In [1]:
import pandas as pd
import os
from functools import reduce
import numpy as np

pd.set_option('display.max_columns', None)

In [2]:
root_url = 'data/medical/2025'
sheet_filepaths = sorted(
    [
        os.path.join(root_url, f) 
        for f in os.listdir(root_url) 
        if f.endswith('.xlsx')
    ]
)

months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

In [3]:
def create_column_names(original_column_names):
    """Add the _M, _F, _Total, _Percent to all indicators"""
    code_column_names = ['PSGC', 'Area']

    for indicator in original_column_names:
        code_column_names.append(indicator + '_M')
        code_column_names.append(indicator + '_F')
        code_column_names.append(indicator + '_Total')
        code_column_names.append(indicator + '_Percent')

    return code_column_names

In [4]:
COLUMN_ORDER = (
    create_column_names(
        [
            'BCG', 'HEPA_B1', 'CPAB', 
            'DPT_1', 'DPT_2', 'DPT_3',
            'OPV_1', 'OPV_2', 'OPV_3',
            'IPV_1', 'IPV_2',
            'PCV_1', 'PCV_2', 'PCV_3',
            'FIC', 'CIC'
        ]
    )
)

COLUMN_ORDER.insert(2, 'Month')

GEO_COLUMN_ORDER = COLUMN_ORDER.copy()
GEO_COLUMN_ORDER.insert(3, 'Geographic Level')

In [5]:
def extract_monthly_data(
    sheet_filepath,
    cols_to_drop,
    col_indicators,
    months=None,
    is_verbose=False,
):
    """
    Extract and clean monthly data from an Excel file, then combine into one DataFrame.
    """

    # Default to all months if none are provided
    if months is None:
        months = [
            "Jan", "Feb", "Mar", "Apr", "May", "Jun",
            "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
        ]

    table_month_dfs = []

    for month in months:
        if is_verbose:
            print(month)

        # Read the specific month's sheet
        table_month_df = pd.read_excel(
            sheet_filepath,
            sheet_name=month,
            skiprows=6,
            skipfooter=4,
        )

        # Drop unwanted columns by index
        table_month_df.drop(
            columns=table_month_df.columns[cols_to_drop],
            inplace=True,
        )

        # Rename columns based on indicator structure
        table_month_df.columns = create_column_names(col_indicators)

        # Add month identifier
        table_month_df["Month"] = month

        # Store processed DataFrame
        table_month_dfs.append(table_month_df)

    # Combine all months into a single DataFrame
    return pd.concat(table_month_dfs, ignore_index=True)

In [6]:
table_configs = {
    "table1": {
        "sheet_filepath": sheet_filepaths[0],
        "cols_to_drop": [2] + list(range(7, 11)) + list(range(15, 20)),
        "col_indicators": ['BCG', 'HEPA_B1', 'CPAB'],
    },
    "table2": {
        "sheet_filepath": sheet_filepaths[1],
        "cols_to_drop": [2] + list(range(15, 28)),
        "col_indicators": ['DPT_1', 'DPT_2', 'DPT_3'],
    },
    "table3": {
        "sheet_filepath": sheet_filepaths[2],
        "cols_to_drop": [2] + list(range(15, 28)),
        "col_indicators": ['OPV_1', 'OPV_2', 'OPV_3'],
    },
    "table4": {
        "sheet_filepath": sheet_filepaths[3],
        "cols_to_drop": [2] + list(range(11, 20)),
        "col_indicators": ['IPV_1', 'IPV_2'],
    },
    "table5": {
        "sheet_filepath": sheet_filepaths[4],
        "cols_to_drop": [2] + list(range(15, 28)),
        "col_indicators": ['PCV_1', 'PCV_2', 'PCV_3'],
    },
    "table6": {
        "sheet_filepath": sheet_filepaths[5],
        "cols_to_drop": [2] + list(range(11, 20)),
        "col_indicators": ['MCV_1', 'MCV_2'],
    },
    "table6": {
        "sheet_filepath": sheet_filepaths[6],
        "cols_to_drop": [2, 7],
        "col_indicators": ['FIC', 'CIC'],
    },
}

In [7]:
table_dfs = []
for table, config in table_configs.items():
    print(table)
    table_df = extract_monthly_data(**config)
    table_dfs.append(table_df)

table1
table2
table3
table4
table5
table6


In [8]:
# there are misspellings right now
corrections = {
    "PagsanJun": "Pagsanjan",
    "Juniuay": "Janiuay",
    "PinamungaJun": "Pinamungajan",
    "PambuJun": "Pambujan",
    "NauJun": "Naujan",
    "Maguindanao Sur": "Maguindanao del Sur",
}

for table_df in table_dfs:
    table_df.Area = table_df.Area.replace(corrections)

In [9]:
# merge the table dataframes by PSGC, Area, and Month
monthly_df = reduce(
    lambda left, right: pd.merge(
        left, 
        right, 
        on=["PSGC", "Area", "Month"], 
        how="outer",
    ),
    table_dfs,
)

# Impute all * into the median
for col in monthly_df.columns:
    if col not in ["PSGC", "Area", "Month"]:
        # Replace "*" safely without dtype guessing
        monthly_df[col] = monthly_df[col].mask(
            monthly_df[col] == "*", np.nan
        )

        # Explicit conversion
        monthly_df[col] = pd.to_numeric(monthly_df[col], errors="coerce")

        # Fill with median
        monthly_df[col] = monthly_df[col].fillna(monthly_df[col].median())

# set an order for the months
month_order = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]
monthly_df["Month"] = pd.Categorical(
    monthly_df["Month"],
    categories=month_order,
    ordered=True
)

# make sure the PSGC is a string and has 10 digits
monthly_df.PSGC = monthly_df.PSGC.astype(str).str.zfill(10)
monthly_df = monthly_df[COLUMN_ORDER].sort_values(by=['PSGC', 'Month'], ascending=[True, True]).copy()

In [10]:
psgc_df = pd.read_excel('data/income/psgc-1q-2025-publication-datafile.xlsx', sheet_name='PSGC')
psgc_df['10-digit PSGC'] = psgc_df['10-digit PSGC'].astype(str).str.zfill(10)
psgc_df = psgc_df[['10-digit PSGC', 'Geographic Level']].copy()

In [11]:
# update zamboanga_sibugay and alburqueqe's psgc code
psgc_fixes = {
    "Zamboanga Sibugay": "0908300000",
    "Alburquerque": "0701201000",
}

for area, psgc in psgc_fixes.items():
    monthly_df.loc[monthly_df["Area"] == area, "PSGC"] = psgc
    
monthly_psgc_df = monthly_df.merge(psgc_df, left_on='PSGC', right_on='10-digit PSGC', how='left')
monthly_psgc_df = monthly_psgc_df[GEO_COLUMN_ORDER].copy()

# Output Time

In [12]:
# Columns that should stay single (not grouped)
base_cols = ['PSGC', 'Area', 'Month', 'Geographic Level']


def write_with_grouped_headers(df, writer, sheet_name):
    df.to_excel(writer, sheet_name=sheet_name, index=False, startrow=1)
    
    workbook  = writer.book
    worksheet = writer.sheets[sheet_name]
    
    # Formats
    header_main = workbook.add_format({
        'bold': True,
        'align': 'center',
        'valign': 'middle',
        'border': 1,
        'bg_color': '#1D4E79',
        'font_color': '#FFFFFF'
    })

    header_sub = workbook.add_format({
        'bold': True,
        'align': 'center',
        'border': 1,
        'bg_color': '#1D4E79',
        'font_color': '#FFFFFF'
    })
        
    highlight_format = workbook.add_format({
        'bg_color': '#FFF2CC'  # light yellow
    })
    
    # ---- STEP 1: Handle base columns ----
    col_idx = 0
    for col in df.columns:
        if col in base_cols:
            worksheet.merge_range(0, col_idx, 1, col_idx, col, header_main)
            col_idx += 1
    
    # ---- STEP 2: Dynamically group remaining columns ----
    grouped_cols = [c for c in df.columns if c not in base_cols]
    
    # Extract prefix (before last "_")
    from collections import defaultdict
    groups = defaultdict(list)
    
    for col in grouped_cols:
        if '_' in col:
            prefix = '_'.join(col.split('_')[:-1])  # e.g. OPV_M → OPV
            groups[prefix].append(col)
        else:
            groups[col].append(col)
    
    # Write grouped headers
    for group, cols in groups.items():
        start_col = col_idx
        end_col = col_idx + len(cols) - 1
        
        # Top header
        worksheet.merge_range(0, start_col, 0, end_col, group, header_main)
        
        # Subheaders
        for i, col in enumerate(cols):
            sub = col.split('_')[-1] if '_' in col else col
            worksheet.write(1, start_col + i, sub, header_sub)
        
        col_idx += len(cols)
    
    # ---- STEP 3: Highlight rows where PSGC ends with 00000000 ----
    if 'PSGC' in df.columns:
        psgc_col_idx = df.columns.get_loc('PSGC')
        
        n_rows = len(df)
        n_cols = len(df.columns)
        
        # Helper to convert column index → Excel letter (handles AA, AB, etc.)
        def colnum_to_excel(n):
            string = ""
            while n >= 0:
                string = chr(n % 26 + 65) + string
                n = n // 26 - 1
            return string
        
        col_letter = colnum_to_excel(psgc_col_idx)
        
        highlight_format = workbook.add_format({
            'bg_color': '#A6A6A6',
            'bold': True
        })
    
    worksheet.conditional_format(
        2, 0,                     # start row, start col
        n_rows + 1, n_cols - 1,   # end row, end col
        {
            'type': 'formula',
            'criteria': f'=RIGHT(TEXT(${col_letter}3,"0"),8)="00000000"',
            'format': highlight_format
        }
    )
    
    # ---- STEP 4: Formatting ----
    worksheet.freeze_panes(2, 0)
    
    for i, col in enumerate(df.columns):
        worksheet.set_column(i, i, 18)


# ---- WRITE FILE ----
with pd.ExcelWriter('outputs/monthly_immunization.xlsx', engine='xlsxwriter') as writer:
    
    # All data
    write_with_grouped_headers(monthly_psgc_df, writer, 'All_Data')
    
    # Per year
    months = monthly_psgc_df['Month'].unique()
    
    for month in months:
        monthly_df = monthly_psgc_df[
            monthly_psgc_df['Month'] == month
        ]
        
        write_with_grouped_headers(monthly_df, writer, str(month))

In [13]:
monthly_psgc_df.to_csv('monthly_immunization.csv', index=False)